In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def khoitao_psi0(x):
    global sigma_0, k_0, x_0, dx

    # Bo song Gauss
    psi_0 = np.exp(-0.5 * ((x - x_0) / sigma_0)**2) * np.exp(1j * k_0 * x)

    # Chuan hoa ham song
    xacsuat = np.sum(np.abs(psi_0)**2) * dx
    psi_0 = psi_0 / np.sqrt(xacsuat)

    return psi_0


def thenang_1diem(x_i):
    global V_max
    if 0 <= x_i <= 15:
        V_i = 0
    else:
        V_i = V_max
    return V_i

def khoitao_thenang(x):
    N_x = len(x)
    V = np.zeros(N_x)
    # The nang tai tung diem luoi
    for i in range(N_x):
        V[i] = thenang_1diem(x[i])
    return V

def khoitao_x_t(x_min, x_max, t_min, t_max):
    global dx, dt

    N_x = int(round((x_max - x_min) / dx)) + 1
    N_t = int(round((t_max - t_min) / dt)) + 1

    x = np.linspace(x_min, x_max, N_x)
    t = np.linspace(t_min, t_max, N_t)

    return x, t, N_x, N_t

In [ ]:
def khoitao_Hamiltonian(x):
    global dx, hbar, m

    N_x = len(x)
    V = khoitao_thenang(x)

    # He so cua dao ham bac hai trong Hamiltonian
    # H = -hbar^2/(2m) d2/dx2 + V
    C = -hbar**2 / (2 * m * dx**2)

    H = np.zeros((N_x, N_x), dtype=complex)

    # Duong cheo chinh
    for i in range(N_x):
        H[i, i] = -2 * C + V[i]

    # Duong cheo duoi
    for i in range(1, N_x):
        H[i, i - 1] = C

    # Duong cheo tren
    for i in range(N_x - 1):
        H[i, i + 1] = C

    return H

In [ ]:
def giai_Crank_Nicolson(psi_0, H, N_t):
    global dt, hbar

    N_x = len(psi_0)

    # Ma tran don vi
    I = np.eye(N_x, dtype=complex)

    # Cong thuc Crank-Nicolson:
    # (I + i dt H / 2hbar) psi^{n+1}
    # =
    # (I - i dt H / 2hbar) psi^n
    A = I + 1j * dt * H / (2 * hbar)
    B = I - 1j * dt * H / (2 * hbar)

    psi = np.zeros((N_t, N_x), dtype=complex)

    # Gan dieu kien ban dau
    psi[0, :] = psi_0

    # Lap theo thoi gian
    for n in range(N_t - 1):
        ve_phai = B @ psi[n, :]

        # Giai he A psi_moi = ve_phai
        psi[n + 1, :] = np.linalg.solve(A, ve_phai)

    return psi

In [ ]:
def tinh_xacsuat(psi):
    global dx

    N_t = psi.shape[0]
    P = np.zeros(N_t)
    

    for n in range(N_t):
        P[n] = np.sum(np.abs(psi[n, :])**2) * dx

    return P

In [5]:
sigma_0 = 0.5
dx = 0.05
dt = 1/4*dx**2
k_0 = 8
x_0 = 5

hbar = 1
m = 1

t_min = 0
t_max = 1.2

x_min = 0
x_max = 15

V_max = 500